In [ ]:
import stim

import sinter

from stimbposd import BPOSD, sinter_decoders

import numpy as np

import matplotlib.pyplot as plt

import multiprocessing

from pathlib import Path

In [ ]:
import sys
from pathlib import Path

if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

from non_circuit_library import threshold as _circuit_library

_circuit_library.configure(3)

from non_circuit_library.threshold import (
    stabilizers_k_z,
    stabilizers_k_x,
    V_even,
    V_odd,
    stabilizers_general,
    measure_logical_qubits_3D_Z,
    measure_logical_qubits_3D_X,
    S_Z_Gate,
    S_Z_DAG_Gate,
    DISTANCE,
    Stabilizers_measurement_general,
)


In [ ]:
def circuit_generate(rate_mea, rate_idle):

    data_qubit_range = list(range(0, 15)) 

    c = stim.Circuit(); c.append("H", data_qubit_range)
    c += S_Z_Gate()
    c += Stabilizers_measurement_general(0, 1)
    c.append("TICK")


    
    c.append("DEPOLARIZE1", data_qubit_range, rate_idle) 
    c += Stabilizers_measurement_general(rate_mea, 2)
    c.append("TICK")

    c.append("DEPOLARIZE1", data_qubit_range, rate_idle) 
    c += Stabilizers_measurement_general(rate_mea, 3)
    c.append("TICK")
    
    c.append("DEPOLARIZE1", data_qubit_range, rate_idle) 
    c += Stabilizers_measurement_general(rate_mea, 4)
    c.append("TICK")
    
    c.append("DEPOLARIZE1", data_qubit_range, rate_idle) 
    c += Stabilizers_measurement_general(0, 5)
    c.append("TICK")
        

    c += S_Z_DAG_Gate()
    c += measure_logical_qubits_3D_X()


    return c

# c += measure_logical_XX()

In [ ]:
c = circuit_generate(0.01, 0.01)
# c.diagram("timeline-svg")

In [ ]:
dem = c.detector_error_model()

In [ ]:
def generate_tasks():
    for p in [0.002, 0.004, 0.006, 0.008, 0.01]:
            yield sinter.Task(
                circuit=circuit_generate(p, p),
                json_metadata={'p': p},
            )

In [ ]:
samples = sinter.collect(
    num_workers=multiprocessing.cpu_count() - 1,
    max_shots=1_000_000,
    max_errors=100,
    tasks=generate_tasks(),
    decoders=["hypergraph_union_find"],
    # custom_decoders=sinter_decoders(),
    save_resume_filepath="3D_color_d3_union_find.csv"
)

In [ ]:
fig, ax = plt.subplots(1, 1)
sinter.plot_error_rate(  
    ax=ax,  
    stats=samples,  
    group_func=lambda stat: stat.decoder,  # No 'd' available  
    x_func=lambda stat: stat.json_metadata['p']  # Placeholder x since 'p' is unavailable  
)
ax.loglog()
ax.grid()
ax.set_title("Logical Error Rate vs Physical Error Rate")
ax.set_ylabel("Logical Error Probability (per shot)")
ax.set_xlabel("Physical Error Rate")
ax.legend()
plt.show()